In [ ]:
import sagemaker
from sklearn.model_selection import train_test_split
import boto3
import pandas as pd
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

import dotenv
dotenv.load_dotenv()

In [ ]:
PROFILE_NAME = os.getenv("AWS_PROFILE")
REGION = os.getenv("AWS_REGION")
BUCKET = os.getenv("BUCKET")
ROLE = os.getenv("ROLE")
FRAMEWORK_VERSION = "0.23-1"

In [4]:
# Sagemaker client
boto3.setup_default_session(profile_name=PROFILE_NAME, region_name=REGION)
sm_boto3 = boto3.client("sagemaker")
bucket = BUCKET
sess = sagemaker.Session(default_bucket=bucket) 
region = sess.boto_session.region_name

In [5]:
# Reading Dataset
'''
import kagglehub
# Download latest version
path = kagglehub.dataset_download("ziyuefu1/bmw-global-sales2018-2025")
print(os.listdir(path))
df = pd.read_csv(os.path.join(path,"bmw_global_sales_2018_2025.csv"))
'''

df = pd.read_csv(r"data\bmw_global_sales_2018_2025.csv")

In [6]:
# Encode cat columns
le = LabelEncoder()
df["Region_enc"] = le.fit_transform(df["Region"])
df["Model_enc"] = le.fit_transform(df["Model"])

# Features e target
X = df[["Year", "Month", "Region_enc", "Model_enc", "Units_Sold", "BEV_Share", "Premium_Share", "GDP_Growth", "Fuel_Price_Index"]]
y = df["Avg_Price_EUR"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
X_train.to_csv("train-V-1.csv", index=False)
X_test.to_csv("test-V-1.csv", index=False)

In [8]:
sk_prefix = "sagemaker/bmw_price_classification/sklearncontainer"
trainpath = sess.upload_data(
    path = "train-V-1.csv", bucket=bucket, key_prefix = sk_prefix
)
testpath = sess.upload_data(
    path = "test-V-1.csv", bucket=bucket, key_prefix = sk_prefix
)

print(trainpath)
print(testpath)

s3://bucketsagemaker-firsttest/sagemaker/bmw_price_classification/sklearncontainer/train-V-1.csv
s3://bucketsagemaker-firsttest/sagemaker/bmw_price_classification/sklearncontainer/test-V-1.csv


In [9]:
%%writefile script.py

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import sklearn
import joblib
import boto3
import pathlib
from io import StringIO
import argparse
import joblib
import os
import numpy as np
import pandas as pd

# Tis function is necessary for loading your model 
def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir,"model.joblib"))
    return clf

if __name__ == "__main__":
    print("[INFO] Extracting arguments")
    parser = argparse.ArgumentParser()

    #hyperparameters sent by the client are passed as command-line arguments
    parser.add_argument("--n_estimators", type=int, default=100)
    parser.add_argument("--random_state", type=int, default=0)

    #Data, model and output directories
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    parser.add_argument("--train-file", type=str, default="train-V-1.csv")
    parser.add_argument("--test-file", type=str, default="test-V-1.csv")

    args, _ = parser.parse_known_args()

    print("SKLearn Version", sklearn.__version__)
    print("joblib Version", joblib.__version__)

    print("[INFO] Reading data")
    print()
    train_df = pd.read_csv(os.path.join(args.train, args.train_file))
    test_df = pd.read_csv(os.path.join(args.test, args.test_file))

    features = list(train_df.columns)
    label = features.pop(-1)

    print("Building training and testing dataset")
    print()
    X_train = train_df[features]
    X_test = test_df[features]
    y_train = train_df[label]
    y_test = test_df[label]

    print("Column order: ")
    print(features)
    print()

    print("Label column is: ", label)
    print()

    print("Data Shape: ")
    print()
    print("--- SHAPE OF TRAINING DATA (85%) --- ")
    print(X_train.shape)
    print(y_train.shape)
    print()
    print("--- SHAPE OF TESTING DATA (15%) --- ")
    print(X_test.shape)
    print(y_test.shape)
    print()

    print("Training RandomForest Model")
    print()
    model = RandomForestRegressor(n_estimators=args.n_estimators, random_state=args.random_state)
    model.fit(X_train,y_train)
    print()

    model_path = os.path.join(args.model_dir, "model.joblib") 
    joblib.dump(model, model_path)
    print("Model persisted at " + model_path)
    print()

    y_pred = model.predict(X_test)
    test_err = mean_absolute_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    print()
    print("--- METRIC RESULTS FOR TESTING DATA ---")
    print()
    print("Total Rows are: ", X_test.shape[0])
    print(f"MAE:  {test_err:,.0f} EUR")
    print(f"R²:   {test_r2:.4f}")

Overwriting script.py


In [ ]:
# Initializing estimator
from sagemaker.sklearn.estimator import SKLearn

sklearn_estimator = SKLearn(
    entry_point="script.py",
    role = ROLE,
    instance_count = 1,
    instance_type="ml.m5.large", 
    framework_version=FRAMEWORK_VERSION,
    base_job_name="RF-custom-sklearn",
    hyperparameters={
        "n_estimators": 10,
        "random_state": 0
    },
    use_spot_instances = False,
    max_run=3600 
)

In [ ]:
# Launching estimator async
sklearn_estimator.fit(
    {"train": trainpath, "test": testpath},
    wait=False,
    logs=False
)

In [ ]:
config_path = os.path.expanduser("~\\.sagemaker-code-config")
with open(config_path, 'w') as f:
    import json
    json.dump({}, f)

os.environ["SAGEMAKER_CODE_CONFIG_PATH"] = config_path

In [ ]:
sklearn_estimator.latest_training_job.wait(logs="None")
artifact = sm_boto3.describe_training_job(
    TrainingJobName=sklearn_estimator.latest_training_job.name
)["ModelArtifacts"]["S3ModelArtifacts"]

print("Model artifact persisted at " + artifact)

In [ ]:
from sagemaker.sklearn.model import SKLearnModel
from time import gmtime, strftime

model_name = "Custom-sklearn-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime()) 
model = SKLearnModel(
    name = model_name,
    model_data = artifact,
    role = ROLE,
    entry_point="script.py",
    framework_version=FRAMEWORK_VERSION
)

In [ ]:
endpoint_name = "Custom-sklearn-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime()) 
print(f"Endpoint name:{endpoint_name}")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
    endpoint_name=endpoint_name
)

In [20]:
predictor

In [ ]:
X_input = X.drop(columns=["Fuel_Price_Index"])
print(predictor.predict(X_input[0:2].values.tolist()))

[0.998 0.996]


In [ ]:
sm_boto3.delete_endpoint(EndpointName = endpoint_name)